In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [6]:
from pprint import pprint

pprint(response)

pprint(tools)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='b9636ffd-91a3-4457-81ab-c2f872c17c0d'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1032, 'prompt_tokens': 1229, 'total_tokens': 2261, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DTueOIqZzb1ZJoP4uqeL63BPHil1R', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8323-ae86-7a00-9bde-0c96d298ddb4-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2027', 'departureDateFlexRange'

In [7]:
print(response["messages"][-1].content)

Direct flight not available on March 31 from San Francisco (SFO) to Tokyo (TYO) in the current results. Here are the best 1-stop options (same-day dates) priced at the cheapest rate:

| Route (with layovers) | Times (local) | Cabin | Price (USD) | Book link |
|---|---|---|---:|---|
| SFO → ICN → NRT | 03/31 01:05 → 04/02 12:40 (32h 0m) | Economy | 936 | https://on.kiwi.com/vx6sHW |
| SFO → ICN → NRT | 03/31 01:05 → 04/02 15:25 (34h 45m) | Economy | 936 | https://on.kiwi.com/ZQrry2 |
| SFO → ICN → NRT | 03/31 12:40 → 04/02 19:45 (39h 5m) | Economy | 936 | https://on.kiwi.com/bK3qoh |

Summary
- Best prices: 936 USD for three 1-stop options via ICN to NRT.
- Shortest among the cheapest options: 32 hours total travel time.
- Recommendation: If you’re set on a budget, the 936 USD options via ICN are your best bet. If you need shorter travel time, consider adjusting dates or exploring other nearby airports; there aren’t any direct SFO→TYO flights shown for March 31 in these results.
- Fun f